# Week 11, Day 4: Exceptions: try and except

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w011d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
values = ["12", "7", "x", "30"]
total = 0
for v in values:
    if v.isdigit():
        total += int(v)
print(total)

In [ ]:
prices = {"tea": 150}
print("Before")
print(prices["coffee"])
print("After")

In [ ]:
text = "12x"
try:
    n = int(text)
    print("Got", n)
except ValueError:
    n = 0
    print("Not a number, using 0")
print("n is", n)

> **Key idea.** `try:` the code that might fail. `except SomeError:` what to do if it raises `SomeError`. The rest of the `try` block is skipped as soon as an error happens.

*Interactive figure on the website: Choose an input. Green lines ran, the red line raised an error, faded lines were skipped. Each except line is checked in turn until one matches..*

In [ ]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None
    except TypeError:
        return "numbers only"

print(safe_divide(10, 4), safe_divide(1, 0), safe_divide("x", 2))

In [ ]:
for text in ["42", "forty-two", "4.2"]:
    try:
        print(int(text))
    except ValueError as e:
        print("Problem:", e)

In [ ]:
def read_number(text):
    return float(text.replace(",", ""))

def total(texts):
    return sum(read_number(t) for t in texts)

try:
    print(total(["1,200", "300.5", "lots"]))
except ValueError as e:
    print("Couldn't add them up:", e)

In [ ]:
def load_lines(name):
    try:
        with open(name) as f:
            return f.read().splitlines()
    except FileNotFoundError:
        return []

print(load_lines("missing.txt"))

In [ ]:
for text in ["8", "zero"]:
    try:
        n = int(text)
    except ValueError:
        print(repr(text), "isn't a number")
    else:
        print(repr(text), "doubled is", n * 2)
    finally:
        print("checked", repr(text))

> **Tip.** You'll rarely need `finally` for files, because `with` already closes them. It's for other things that must be undone: putting a setting back, releasing a lock, stopping a timer.

In [ ]:
while True:
    text = input("How many tickets? ")
    try:
        n = int(text)
    except ValueError:
        print("Please type a whole number.")
        continue
    if n < 1:
        print("At least one, please.")
        continue
    break
print("Booking", n, "tickets")

In [ ]:
def ask_int(prompt, low, high):
    while True:
        try:
            n = int(input(prompt))
        except ValueError:
            print("That isn't a whole number.")
            continue
        if low <= n <= high:
            return n
        print(f"Choose from {low} to {high}.")

age = ask_int("Age: ", 0, 130)
print("Age", age)

In [ ]:
data = {"tea": 150}
try:
    price = dta["tea"]
except Exception:
    price = 0
print("Price:", price)

> **Common mistake.** Catch the **specific** exception you expect, and only around the line where you expect it. A program that crashes on a bug is far better than one that hides it and gives a wrong answer.

### Exercise: Safe conversion

Write `to_number(text)` that returns `text` converted to a float, or `None` if it can't be. Use `try` and `except ValueError`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several texts\"\n    assert to_number(\"3.5\") == 3.5\n    assert to_number(\" -2 \") == -2.0\n    assert to_number(\"1e3\") == 1000.0\n    assert to_number(\"abc\") is None\n    assert to_number(\"\") is None\n    assert \"except ValueError\" in source")

### Exercise: Add what you can

Write `sum_valid(texts)` that adds up the strings that are valid numbers and returns a tuple `(total, number_skipped)`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several lists\"\n    assert sum_valid([\"4\", \"x\", \"2.5\", \"\", \"-1\"]) == (5.5, 2)\n    assert sum_valid([]) == (0, 0)")

### Exercise: How many tickets?

Ask `How many tickets? ` until the answer is a whole number from 1 to 10. Print `Please type a whole number.` for something that isn't one, and `From 1 to 10, please.` for a number out of range. Then print `Booked N tickets`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several attempts\"\n    expect_output(\"How many tickets? two\\nPlease type a whole number.\\nHow many tickets? 0\\nFrom 1 to 10, please.\\nHow many tickets? 11\\nFrom 1 to 10, please.\\nHow many tickets? 3\\nBooked 3 tickets\", inputs=[\"two\", \"0\", \"11\", \"3\"], echo=True)\n    expect_output(\"How many tickets? 10\\nBooked 10 tickets\", inputs=[\"10\"], echo=True)")

### Exercise: Price or message

Write `price_message(prices, item)` that returns `"tea costs 150"`, or `"Sorry, no juice"` if the item isn't in the dictionary. Use `try` and `except KeyError`, with `else` for the success case.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several items\"\n    p = {\"tea\": 150, \"coffee\": 900}\n    assert price_message(p, \"tea\") == \"tea costs 150\"\n    assert price_message(p, \"juice\") == \"Sorry, no juice\"\n    assert \"else:\" in source and \"except KeyError\" in source")

### Exercise: First line, if any

Write `first_line(name)` that returns the first line of a file without its newline, `"(empty file)"` if the file is empty, or `"(no such file)"` if it doesn't exist.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several files\"\n    write_file(\"a.txt\", \"hello\\nworld\\n\")\n    write_file(\"e.txt\", \"\")\n    assert first_line(\"a.txt\") == \"hello\"\n    assert first_line(\"e.txt\") == \"(empty file)\"\n    assert first_line(\"zzz.txt\") == \"(no such file)\"")

### Exercise: Safe index

Write `get(items, i, default)` that returns `items[i]`, or `default` if the index is out of range or `items` can't be indexed by `i` (for example, `i` is a string). Catch both kinds of error in one `except`.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several calls\"\n    assert get([1, 2, 3], 1, 0) == 2\n    assert get([1, 2, 3], 9, 0) == 0\n    assert get([1, 2, 3], -1, 0) == 3\n    assert get([1, 2, 3], \"x\", \"?\") == \"?\"\n    assert \"IndexError\" in source and \"TypeError\" in source and source.count(\"except\") == 1, \"Catch both in one except, with a tuple.\"")

### Bug hunt: It hides the bug

`average` should give the average of a list, or `None` for an empty list. It always returns `None`, because of a typo that the `except` is hiding. Make the `except` catch only the error it's meant for, then fix the typo it reveals.

In [ ]:
def average(values):
    try:
        return sum(values) / len(value)
    except Exception:
        return None

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"averages\"\n    assert average([2, 4]) == 3\n    assert average([]) is None\n    assert \"except Exception\" not in source and \"except ZeroDivisionError\" in source")

### Bug hunt: Every number is bad

This should print `Bad number: x`, then `Total: 12`. Instead it says every value is bad, and the total is 0. Keep only the conversion inside `try`, move the adding into an `else` block, catch only `ValueError`, and then fix the bug that this reveals.

In [ ]:
total = 0
for text in ["5", "x", "7"]:
    try:
        n = int(text)
        total = totl + n
    except Exception:
        print("Bad number:", text)
print("Total:", total)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"output\"\n    expect_output(\"Bad number: x\\nTotal: 12\")\ndef test_shape():\n    \"only the conversion is tried\"\n    assert \"else:\" in source and \"except ValueError\" in source and \"Exception\" not in source")

### Bug hunt: The wrong error

This should print `No such student`, but it crashes with a `KeyError`. Fix the `except`.

In [ ]:
marks = {"Ada": 93}
try:
    print(marks["Bo"])
except IndexError:
    print("No such student")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"handled\"\n    expect_output(\"No such student\")")

### Puzzle: Read a number from a file

Arrange the lines so that `read_count` returns the number in a file, or 0 if the file is missing or doesn't hold a number. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work out what each program prints, and write it as a string (use `\n` between lines).

In [ ]:
# Program A
#   try:
#       print(1)
#       print(1 / 0)
#       print(2)
#   except ZeroDivisionError:
#       print(3)
#   print(4)
a = ___
# Program B
#   try:
#       x = int("5")
#   except ValueError:
#       print("bad")
#   else:
#       print("good", x)
#   finally:
#       print("end")
b = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each output\"\n    assert a == \"1\\n3\\n4\"\n    assert b == \"good 5\\nend\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a forgiving calculator

Write `calculate(line)` for a calculator that takes lines like `12 * 3.5` (a number, a space, an operator `+ - * / ** %`, a space, a number) and returns the answer as a string, or a clear message:

- `Bad format` if the line isn't three parts;
- `Not a number: TEXT` for the first part that isn't a number;
- `Unknown operator: OP` for any other operator;
- `Can't divide by zero` for `/` or `%` by zero;
- `Too big` if the answer is too large for a float (`2 ** 99999.0` raises `OverflowError`).

Answers that are whole numbers are shown without `.0`: `2 * 3` gives `6`, `7 / 2` gives `3.5`. Then write a loop that reads lines with the prompt `> ` until `quit`, printing each result.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"answers\"\n    assert calculate(\"12 * 3.5\") == \"42\"\n    assert calculate(\"7 / 2\") == \"3.5\"\n    assert calculate(\"2 ** 10\") == \"1024\"\n    assert calculate(\"-5 % 3\") == \"1\"\ndef test_errors():\n    \"messages\"\n    assert calculate(\"1 +\") == \"Bad format\"\n    assert calculate(\"1 + 2 + 3\") == \"Bad format\"\n    assert calculate(\"x + 2\") == \"Not a number: x\"\n    assert calculate(\"2 + two\") == \"Not a number: two\"\n    assert calculate(\"2 ^ 3\") == \"Unknown operator: ^\"\n    assert calculate(\"5 / 0\") == \"Can't divide by zero\"\n    assert calculate(\"5 % 0\") == \"Can't divide by zero\"\n    assert calculate(\"10 ** 99999\") == \"Too big\"\ndef test_loop():\n    \"the loop\"\n    expect_output(\"> 1 + 1\\n2\\n> 1 / 0\\nCan't divide by zero\\n> quit\", inputs=[\"1 + 1\", \"1 / 0\", \"quit\"], echo=True)")

## Warm-up

Checking with `isdigit` first works for whole numbers. But what about `"-4"`, `"2.5"` or `" 7 "`? `int` and `float` accept those, `isdigit` doesn't. It's often easier to **try** the conversion and deal with failure if it happens. That's today's tool.

## What an exception is

When Python can't carry out an operation, it **raises an exception**: an object describing what went wrong. You've seen many:

| Exception | Raised when | Example |
|---|---|---|
| `ValueError` | the type is right but the value isn't | `int("abc")` |
| `TypeError` | the type is wrong | `"a" + 1` |
| `ZeroDivisionError` | dividing by zero | `5 / 0` |
| `IndexError` | a list index is out of range | `[1, 2][5]` |
| `KeyError` | a dictionary key is missing | `{}["x"]` |
| `FileNotFoundError` | a file to read doesn't exist | `open("nope.txt")` |
| `NameError` | a name isn't defined | `print(totl)` |
| `AttributeError` | an object doesn't have that method | `[1].upper()` |

If nothing **catches** the exception, Python stops the program and prints the **traceback**: the error's type, its message, and the lines that led to it.

`After` never printed. Everything after the error was abandoned.

## try and except

Put code that might fail in a `try` block, and what to do if it fails in an `except` block:

Python runs the `try` block. If a line in it raises a `ValueError`, Python **stops the `try` block at that line**, jumps to the `except` block, runs it, and carries on after the whole statement. If nothing goes wrong, the `except` block is skipped.

### Several kinds of error

A `try` can have several `except` blocks. Python uses the **first** one that matches:

One `except` can catch several types, in a tuple: `except (ValueError, TypeError):`.

### The exception object

`except ValueError as e:` names the exception object `e`. Printing it gives the message Python would have shown:

### Errors from deeper down

An exception travels **up** from the function where it happened, through every function that called it, until something catches it. So you can catch it where you know what to do about it:

The error happened inside `float`, inside `read_number`, inside `total`. The `try` in the main program caught it.

### A missing file

The commonest use of all: a file that might not be there.

## else and finally

Two more optional parts:

- `else:` runs only if the `try` block finished **without** an error.
- `finally:` runs **always**, error or not, even if the `try` block returns. It's for clean-up that must happen whatever.

Why `else` instead of putting the code at the end of the `try` block? Because then an error **in that code** would be caught too, by an `except` that wasn't written for it. Keep the `try` block as small as possible: just the line that might fail.

## Asking until it's right

Combining `while True`, `try`, `except` and `break` gives the standard way to get valid input:

The loop only ends at `break`, which is reached only when the input is a whole number of at least 1. Two kinds of problem, two messages.

Put it in a function and you can reuse it everywhere:

## Catching the right things

### Look before you leap, or ask forgiveness?

There are two styles of dealing with a problem:

- **Look before you leap**: check first, then act. `if key in d: value = d[key]`.
- **Easier to ask forgiveness than permission**: just do it, and handle the failure. `try: value = d[key] except KeyError: ...`.

Python programmers often prefer the second style when failure is unusual, because the check and the action can't get out of step (a file could be deleted between checking it exists and opening it). Use the first style when the check is simple and failure is common, like `d.get(key, 0)`.

### Don't catch everything

A bare `except:`, or `except Exception:`, catches **every** error, including the ones you didn't think of: typos, wrong names, bugs. They'd be silently swallowed:

The real problem is a typo, `dta` instead of `data`, a `NameError`. The program quietly printed a wrong answer. With `except KeyError:` the typo would have crashed the program, and you'd have found it at once.

### Families of exceptions

Exceptions come in families. `KeyError` and `IndexError` are both kinds of `LookupError`; `FileNotFoundError` is a kind of `OSError`; almost every error is a kind of `Exception`. An `except` catches the type it names **and every type in its family**. That's why `except Exception` catches nearly everything.

## Practice

## Quiz

## Daily challenge

## Before you go

Look back at a program you wrote in an earlier week that crashes on bad input, and make it ask again instead. Then explain, in one sentence, why `except Exception` is usually a mistake. Tomorrow: raising your own exceptions, and a method for finding bugs.